In [5]:
import polars as pl
import numpy as np
from sklearn.cluster import KMeans
import pandas as pd
import os


### **Đặc trưng:** Phân khúc loại sản phẩm (category_l1) theo mức giá: Bình dân - Trung cấp - Cao cấp

In [296]:
path_item = r"D:\003. HK1 - Năm 3\02. CS116 - Python cho Máy học\CS116-DoAn\Phase-2\preprocessing data\sale_pers.item_chunk_0.parquet"

df_item = pl.read_parquet(path_item)
print(df_item.shape)
print(df_item.head())


(27323, 11)
shape: (5, 11)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ item_id   ┆ price     ┆ category_ ┆ category_ ┆ … ┆ gender_ta ┆ descripti ┆ brand_fin ┆ age_grou │
│ ---       ┆ ---       ┆ l1        ┆ l2        ┆   ┆ rget_fina ┆ on_final  ┆ al        ┆ p_final  │
│ str       ┆ decimal[3 ┆ ---       ┆ ---       ┆   ┆ l         ┆ ---       ┆ ---       ┆ ---      │
│           ┆ 8,4]      ┆ str       ┆ str       ┆   ┆ ---       ┆ str       ┆ str       ┆ str      │
│           ┆           ┆           ┆           ┆   ┆ str       ┆           ┆           ┆          │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ 050202000 ┆ 99000.000 ┆ Babycare  ┆ Bình sữa, ┆ … ┆ Không xác ┆ Chi tiết  ┆ Dr.Brown' ┆ Từ 9M    │
│ 0004      ┆ 0         ┆           ┆ phụ kiện  ┆   ┆ định      ┆ sản phẩm  ┆ s         ┆          │
│           ┆           ┆           ┆           ┆   ┆           

In [297]:
def cluster_price_group(prices):
    prices_reshaped = prices.to_numpy().reshape(-1, 1)

    # Gom cụm 3 cluster
    kmeans = KMeans(n_clusters=3, random_state=42)
    labels = kmeans.fit_predict(prices_reshaped)

    # Tính mean của từng cluster bằng pandas
    df_temp = pd.DataFrame({"price": prices.values, "cluster": labels})
    cluster_means = df_temp.groupby("cluster")["price"].mean()

    # Sắp xếp cluster theo giá tăng dần → 0,1,2
    sorted_clusters = cluster_means.sort_values().index.tolist()

    # Mapping cluster gốc → 0,1,2
    cluster_map = {sorted_clusters[i]: i for i in range(3)}

    # Trả về danh sách nhãn theo thứ tự index
    return [cluster_map[c] for c in labels]

In [298]:
pdf = df_item.select(["item_id", "price", "category_l1"]).to_pandas()

# Tạo cột rỗng
pdf["price_segment"] = -1

for cat, group in pdf.groupby("category_l1"):
    X = group["price"].values.reshape(-1,1)

    if len(group) < 3:
        # Không gom cụm nếu số lượng ít → gán 0 hết
        pdf.loc[group.index, "price_segment"] = 0
        continue

    # Gom cụm
    kmeans = KMeans(n_clusters=3, random_state=42, n_init="auto")
    labels = kmeans.fit_predict(X)

    # Remap theo thứ tự giá trung bình
    cluster_mean = pd.DataFrame({
        "cluster": labels,
        "price": group["price"].values
    }).groupby("cluster")["price"].mean().sort_values()

    mapping = {cluster: rank for rank, cluster in enumerate(cluster_mean.index)}

    # Gán nhãn theo index
    pdf.loc[group.index, "price_segment"] = [mapping[c] for c in labels]


In [299]:
print(pdf.groupby(["category_l1", "price_segment"]).size())


category_l1             price_segment
Babycare                0                1795
                        1                 165
                        2                  34
Gói Hội Viên            0                   3
                        1                   1
                        2                   3
Hóa mỹ phẩm cho bé      0                 168
                        1                 148
                        2                  29
Hóa mỹ phẩm gia đình    0                 178
                        1                 183
                        2                  26
Phụ kiện                0                1492
                        1                1134
                        2                 521
Sữa                     0                 159
                        1                 205
                        2                  73
Sữa nước                0                 131
                        1                  27
                        2                 

Mapping:
- 0: Bình dân
- 1: Trung cấp
- 2: Cao cấp

In [300]:
segment_stats = (
    pdf.groupby(["category_l1", "price_segment"])["price"]
       .agg(["min", "max", "mean", "median", "count"])
       .sort_values(["category_l1", "price_segment"])
)

print("\n=== BẢNG THỐNG KÊ THEO SEGMENT ===")
print(segment_stats)



=== BẢNG THỐNG KÊ THEO SEGMENT ===
                                               min            max  \
category_l1            price_segment                                
Babycare               0                 1000.0000   1767273.0000   
                       1              1821000.0000   6775000.0000   
                       2              6950000.0000  20990000.0000   
Gói Hội Viên           0                19000.0000     99000.0000   
                       1               169000.0000    169000.0000   
                       2               230000.0000    299000.0000   
Hóa mỹ phẩm cho bé     0                20000.0000    140000.0000   
                       1               145000.0000    290000.0000   
                       2               295000.0000    685000.0000   
Hóa mỹ phẩm gia đình   0                12000.0000    135000.0000   
                       1               139000.0000    275000.0000   
                       2               288000.0000    750000.0000  

In [301]:
df_price_seg = pl.from_pandas(pdf)


In [302]:
df_item = df_item.join(
    df_price_seg.select(["item_id", "price_segment"]),
    on="item_id",
    how="left"
)


In [303]:
print(df_item.head())
print(df_item.select("price_segment").unique())


shape: (5, 12)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ item_id   ┆ price     ┆ category_ ┆ category_ ┆ … ┆ descripti ┆ brand_fin ┆ age_group ┆ price_se │
│ ---       ┆ ---       ┆ l1        ┆ l2        ┆   ┆ on_final  ┆ al        ┆ _final    ┆ gment    │
│ str       ┆ decimal[3 ┆ ---       ┆ ---       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ---      │
│           ┆ 8,4]      ┆ str       ┆ str       ┆   ┆ str       ┆ str       ┆ str       ┆ i64      │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ 050202000 ┆ 99000.000 ┆ Babycare  ┆ Bình sữa, ┆ … ┆ Chi tiết  ┆ Dr.Brown' ┆ Từ 9M     ┆ 0        │
│ 0004      ┆ 0         ┆           ┆ phụ kiện  ┆   ┆ sản phẩm  ┆ s         ┆           ┆          │
│           ┆           ┆           ┆           ┆   ┆ …         ┆           ┆           ┆          │
│ 001029004 ┆ 69000.000 ┆ Thời      ┆ Cơ cấu    ┆ … ┆ Không xác ┆ Con Cưng  

In [304]:
output_path = r"D:\003. HK1 - Năm 3\02. CS116 - Python cho Máy học\CS116-DoAn\Phase-2\Feature engineering\sale_pers.item_chunk_0.parquet"

df_item.write_parquet(output_path)

print(f"Đã lưu thành công vào:\n{output_path}")


Đã lưu thành công vào:
D:\003. HK1 - Năm 3\02. CS116 - Python cho Máy học\CS116-DoAn\Phase-2\Feature engineering\sale_pers.item_chunk_0.parquet


**Nhận Xét:**

**1. Các nhóm ngành có phân khúc giá rất rõ ràng và hợp lý**

Những category_l1 như Sữa, Tã, Textile, Đồ chơi & Sách, TPCN, Hóa mỹ phẩm, Babycare thể hiện phân tách giá rất mạnh:
- Cụm 0 (Bình dân): giá thấp – trung bình
- Cụm 1 (Trung cấp): giá tăng đáng kể
- Cụm 2 (Cao cấp): giá cao vượt trội và rất đặc trưng

**2. Các ngành hàng giá thấp như “Phụ kiện”, “Thực phẩm cho bé”, “Vệ sinh” không quá chênh lệch giữa 0 - 1 - 2**

=> Phân bố không quá rõ ràng để ứng dụng phân cụm

**3. Chỉ có "Tã" và "Sữa" là có phân khúc Trung cấp nhiều hơn Cao cấp**

In [305]:
#===================================

### **Đặc trưng:** Số loại sản phẩm (category_l1) trung bình cho một lần mua hàng (một ngày), và phân cụm số loại giao dịch trung bình đó, gán vào 3 nhãn "Mua ít", "Mua vừa" và "Mua nhiều".

In [306]:
path_tx_0 = r"D:\003. HK1 - Năm 3\02. CS116 - Python cho Máy học\CS116-DoAn\Phase-2\preprocessing data\sale_pers.purchase_history_daily_chunk_0.parquet"

df_tx0 = pl.read_parquet(path_tx_0)

print("Số dòng:", df_tx0.height)
print("Số cột:", df_tx0.width)
print("\nDanh sách cột:")
print(df_tx0.columns)


Số dòng: 1786491
Số cột: 12

Danh sách cột:
['item_id', 'price', 'quantity', 'customer_id', 'created_date', 'channel', 'payment', 'location', 'discount', 'list_price', 'category_l2', 'discount_rate']


In [307]:
df_tx0.head()

item_id,price,quantity,customer_id,created_date,channel,payment,location,discount,list_price,category_l2,discount_rate
str,"decimal[38,4]",i32,i32,date,str,str,i32,"decimal[38,4]","decimal[38,4]",str,"decimal[38,4]"
"""7115000000004""",49000.0000,1,5254214,2024-12-24,"""In-Store""","""VietQR""",656,0.0000,49000.0000,"""Snack ăn dặm""",0.0000
"""0029130000030""",69000.0000,1,7573232,2024-12-24,"""In-Store""","""Tiền mặt""",143,0.0000,74000.0000,"""Bột ăn dặm""",0.0676
"""3496000000053""",75000.0000,2,8187418,2024-12-24,"""In-Store""","""MoMo""",213,0.0000,75000.0000,"""Quần áo & Phụ kiện sơ sinh""",0.0000
"""2700000000002""",58500.0000,2,8187418,2024-12-24,"""In-Store""","""MoMo""",213,13000.0000,65000.0000,"""Khăn khô""",0.1000
"""0029110000036""",89000.0000,1,6931560,2024-12-28,"""Android""","""MoMo""",590,10000.0000,99000.0000,"""Snack ăn dặm""",0.1010


In [308]:
# Thư mục chứa các file transaction 0..19
TX_DIR = r"D:\003. HK1 - Năm 3\02. CS116 - Python cho Máy học\CS116-DoAn\Phase-2\preprocessing data"
TX_PATTERN = "sale_pers.purchase_history_daily_chunk_{}.parquet"

# File item đã preprocessing / feature engineering
ITEM_PATH = r"D:\003. HK1 - Năm 3\02. CS116 - Python cho Máy học\CS116-DoAn\Phase-2\Feature engineering\sale_pers.item_chunk_0.parquet"


In [309]:
# Đọc item
df_item = pl.read_parquet(ITEM_PATH)
print("Item shape:", df_item.shape)

# Chỉ giữ các cột cần thiết cho đặc trưng này
df_item_small = df_item.select(["item_id", "category_l1"])
print(df_item_small.head())


Item shape: (27323, 12)
shape: (5, 2)
┌───────────────┬────────────────┐
│ item_id       ┆ category_l1    │
│ ---           ┆ ---            │
│ str           ┆ str            │
╞═══════════════╪════════════════╡
│ 0502020000004 ┆ Babycare       │
│ 0010290040150 ┆ Thời trang     │
│ 0008010000015 ┆ Đồ chơi & Sách │
│ 0020010000094 ┆ Tã             │
│ 0020010000098 ┆ Tã             │
└───────────────┴────────────────┘


In [310]:
customer_agg_chunks = []  # list để lưu aggregate theo chunk

for i in range(20):  # từ 0 đến 19
    tx_path = os.path.join(TX_DIR, TX_PATTERN.format(i))
    print(f"\n=== Đang xử lý chunk {i}: {tx_path} ===")
    
    # Đọc tối thiểu 3 cột
    tx_chunk = pl.read_parquet(
        tx_path,
        columns=["item_id", "customer_id", "created_date"]
    )
    print("  Transaction chunk shape:", tx_chunk.shape)
    
    # Join với item để lấy category_l1
    tx_joined = tx_chunk.join(
        df_item_small,
        on="item_id",
        how="left"
    )
    
    # Group theo (customer_id, created_date) để đếm số loại category_l1 / ngày
    daily_cat_count = (
        tx_joined
        .group_by(["customer_id", "created_date"])
        .agg(
            pl.col("category_l1").n_unique().alias("unique_categories")
        )
    )
    
    print("  daily_cat_count shape:", daily_cat_count.shape)
    
    # Aggregate theo customer_id trong từng chunk:
    # - tổng số loại
    # - số ngày mua hàng
    cust_chunk = (
        daily_cat_count
        .group_by("customer_id")
        .agg([
            pl.col("unique_categories").sum().alias("sum_unique_categories"),
            pl.len().alias("num_days")
        ])
    )
    
    print("  cust_chunk shape:", cust_chunk.shape)
    
    customer_agg_chunks.append(cust_chunk)

print("\n=== Hoàn thành xử lý tất cả chunk transaction ===")



=== Đang xử lý chunk 0: D:\003. HK1 - Năm 3\02. CS116 - Python cho Máy học\CS116-DoAn\Phase-2\preprocessing data\sale_pers.purchase_history_daily_chunk_0.parquet ===
  Transaction chunk shape: (1786491, 3)
  daily_cat_count shape: (768604, 3)
  cust_chunk shape: (486150, 3)

=== Đang xử lý chunk 1: D:\003. HK1 - Năm 3\02. CS116 - Python cho Máy học\CS116-DoAn\Phase-2\preprocessing data\sale_pers.purchase_history_daily_chunk_1.parquet ===
  Transaction chunk shape: (1786491, 3)
  daily_cat_count shape: (732049, 3)
  cust_chunk shape: (464722, 3)

=== Đang xử lý chunk 2: D:\003. HK1 - Năm 3\02. CS116 - Python cho Máy học\CS116-DoAn\Phase-2\preprocessing data\sale_pers.purchase_history_daily_chunk_2.parquet ===
  Transaction chunk shape: (1786491, 3)
  daily_cat_count shape: (741831, 3)
  cust_chunk shape: (482378, 3)

=== Đang xử lý chunk 3: D:\003. HK1 - Năm 3\02. CS116 - Python cho Máy học\CS116-DoAn\Phase-2\preprocessing data\sale_pers.purchase_history_daily_chunk_3.parquet ===
  Tra

In [311]:
# Gộp tất cả aggregate từng chunk
customer_agg_all = pl.concat(customer_agg_chunks, how="vertical")
print("customer_agg_all shape:", customer_agg_all.shape)

# Group lần cuối theo customer_id để cộng dồn các chunk
customer_final = (
    customer_agg_all
    .group_by("customer_id")
    .agg([
        pl.col("sum_unique_categories").sum().alias("total_unique_categories"),
        pl.col("num_days").sum().alias("total_days")
    ])
)

# Tính trung bình số loại category_l1 mỗi ngày
customer_final = customer_final.with_columns(
    (pl.col("total_unique_categories") / pl.col("total_days")).alias("avg_categories_per_day")
)

print("\n=== Bảng đặc trưng trung bình số loại category_l1 / ngày ===")
print(customer_final.head())
print("Số lượng khách hàng:", customer_final.height)


customer_agg_all shape: (9875793, 3)

=== Bảng đặc trưng trung bình số loại category_l1 / ngày ===
shape: (5, 4)
┌─────────────┬─────────────────────────┬────────────┬────────────────────────┐
│ customer_id ┆ total_unique_categories ┆ total_days ┆ avg_categories_per_day │
│ ---         ┆ ---                     ┆ ---        ┆ ---                    │
│ i32         ┆ u32                     ┆ u32        ┆ f64                    │
╞═════════════╪═════════════════════════╪════════════╪════════════════════════╡
│ 7912392     ┆ 1                       ┆ 1          ┆ 1.0                    │
│ 7396997     ┆ 1                       ┆ 1          ┆ 1.0                    │
│ 7279939     ┆ 40                      ┆ 25         ┆ 1.6                    │
│ 690290      ┆ 2                       ┆ 2          ┆ 1.0                    │
│ 6323569     ┆ 1                       ┆ 1          ┆ 1.0                    │
└─────────────┴─────────────────────────┴────────────┴────────────────────────┘
Số lượn

In [312]:
customer_final = customer_final.with_columns(
    (pl.col("total_unique_categories") / pl.col("total_days")).alias("avg_categories_per_day")
)

print("\n=== 10 khách hàng ngẫu nhiên để kiểm tra kết quả trung gian ===")

sample_10 = (
    customer_final
    .select(["customer_id", "total_unique_categories", "total_days", "avg_categories_per_day"])
    .sample(n=10, shuffle=True)
)

print(sample_10)




=== 10 khách hàng ngẫu nhiên để kiểm tra kết quả trung gian ===
shape: (10, 4)
┌─────────────┬─────────────────────────┬────────────┬────────────────────────┐
│ customer_id ┆ total_unique_categories ┆ total_days ┆ avg_categories_per_day │
│ ---         ┆ ---                     ┆ ---        ┆ ---                    │
│ i32         ┆ u32                     ┆ u32        ┆ f64                    │
╞═════════════╪═════════════════════════╪════════════╪════════════════════════╡
│ 7736620     ┆ 3                       ┆ 3          ┆ 1.0                    │
│ 688402      ┆ 1                       ┆ 1          ┆ 1.0                    │
│ 6493366     ┆ 2                       ┆ 1          ┆ 2.0                    │
│ 5063621     ┆ 26                      ┆ 20         ┆ 1.3                    │
│ 4094962     ┆ 1                       ┆ 1          ┆ 1.0                    │
│ 8202527     ┆ 1                       ┆ 1          ┆ 1.0                    │
│ 4369401     ┆ 3                       

| Cột                       | Ý nghĩa                                                  |
| ------------------------- | -------------------------------------------------------- |
| `total_unique_categories` | Tổng số loại category_l1 khách mua trong toàn bộ lịch sử |
| `total_days`              | Số ngày khách có mua hàng                                |
| `avg_categories_per_day`  | Số loại category_l1 trung bình mỗi ngày khách mua        |


In [313]:
# Lấy dữ liệu để gom cụm
X = customer_final["avg_categories_per_day"].to_numpy().reshape(-1, 1)

print("\nGiá trị min/max của avg_categories_per_day:")
print("  Min:", X.min(), " Max:", X.max())

# KMeans 3 cụm
kmeans = KMeans(n_clusters=3, random_state=42, n_init="auto")
labels = kmeans.fit_predict(X)

# Thêm nhãn cụm vào bảng
customer_final = customer_final.with_columns(
    pl.Series("buy_segment_raw", labels)
)



Giá trị min/max của avg_categories_per_day:
  Min: 1.0  Max: 11.0


In [314]:
# === Tính mean avg_categories_per_day theo cụm ===
cluster_stats = (
    customer_final
    .group_by("buy_segment_raw")
    .agg(
        pl.col("avg_categories_per_day").mean().alias("mean_avg_categories")
    )
    .sort("mean_avg_categories")
)

print("\n=== Mean avg_categories_per_day theo cụm KMeans (trước remap) ===")
print(cluster_stats)

# === Tạo mapping từ cluster gốc -> cluster mới (0: ít, 1: vừa, 2: nhiều) ===
mapping = {}
for new_label, row in enumerate(cluster_stats.iter_rows(named=True)):
    orig_cluster = row["buy_segment_raw"]
    mapping[orig_cluster] = new_label

print("\nMapping cụm:")
for orig, new in mapping.items():
    print(f"  Cluster {orig} -> {new}")

# === Áp dụng mapping ===
customer_final = customer_final.with_columns(
    pl.col("buy_segment_raw").replace(mapping).alias("buy_segment")
)

print("\n=== Kết quả cuối cùng (mẫu) ===")
print(customer_final.select(["customer_id", "avg_categories_per_day", "buy_segment"]).head(10))



=== Mean avg_categories_per_day theo cụm KMeans (trước remap) ===
shape: (3, 2)
┌─────────────────┬─────────────────────┐
│ buy_segment_raw ┆ mean_avg_categories │
│ ---             ┆ ---                 │
│ i32             ┆ f64                 │
╞═════════════════╪═════════════════════╡
│ 0               ┆ 1.070549            │
│ 2               ┆ 1.848737            │
│ 1               ┆ 3.419495            │
└─────────────────┴─────────────────────┘

Mapping cụm:
  Cluster 0 -> 0
  Cluster 2 -> 1
  Cluster 1 -> 2

=== Kết quả cuối cùng (mẫu) ===
shape: (10, 3)
┌─────────────┬────────────────────────┬─────────────┐
│ customer_id ┆ avg_categories_per_day ┆ buy_segment │
│ ---         ┆ ---                    ┆ ---         │
│ i32         ┆ f64                    ┆ i32         │
╞═════════════╪════════════════════════╪═════════════╡
│ 7912392     ┆ 1.0                    ┆ 0           │
│ 7396997     ┆ 1.0                    ┆ 0           │
│ 7279939     ┆ 1.6                    ┆ 1 

In [315]:
segment_check = (
    customer_final
    .group_by("buy_segment")
    .agg([
        pl.count().alias("num_customers"),
        pl.col("avg_categories_per_day").min().alias("min_avg"),
        pl.col("avg_categories_per_day").median().alias("median_avg"),
        pl.col("avg_categories_per_day").mean().alias("mean_avg"),
        pl.col("avg_categories_per_day").max().alias("max_avg"),
    ])
    .sort("buy_segment")
)

print("\n=== Thống kê theo từng segment mua (ít = 0 / vừa = 1 / nhiều = 2) ===")
print(segment_check)



=== Thống kê theo từng segment mua (ít = 0 / vừa = 1 / nhiều = 2) ===
shape: (3, 6)
┌─────────────┬───────────────┬──────────┬────────────┬──────────┬──────────┐
│ buy_segment ┆ num_customers ┆ min_avg  ┆ median_avg ┆ mean_avg ┆ max_avg  │
│ ---         ┆ ---           ┆ ---      ┆ ---        ┆ ---      ┆ ---      │
│ i32         ┆ u32           ┆ f64      ┆ f64        ┆ f64      ┆ f64      │
╞═════════════╪═══════════════╪══════════╪════════════╪══════════╪══════════╡
│ 0           ┆ 1488669       ┆ 1.0      ┆ 1.0        ┆ 1.070549 ┆ 1.459459 │
│ 1           ┆ 818010        ┆ 1.459854 ┆ 1.882353   ┆ 1.848737 ┆ 2.633333 │
│ 2           ┆ 135627        ┆ 2.634146 ┆ 3.0        ┆ 3.419495 ┆ 11.0     │
└─────────────┴───────────────┴──────────┴────────────┴──────────┴──────────┘


C:\Users\PC\AppData\Local\Temp\ipykernel_12820\1373689411.py:5: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  pl.count().alias("num_customers"),


In [316]:
for seg in [0, 1, 2]:
    print(f"\n--- Ví dụ khách hàng thuộc segment {seg} ---")
    sample = (
        customer_final
        .filter(pl.col("buy_segment") == seg)
        .select(["customer_id", "avg_categories_per_day", "buy_segment"])  
        .head(10)
    )
    print(sample)



--- Ví dụ khách hàng thuộc segment 0 ---
shape: (10, 3)
┌─────────────┬────────────────────────┬─────────────┐
│ customer_id ┆ avg_categories_per_day ┆ buy_segment │
│ ---         ┆ ---                    ┆ ---         │
│ i32         ┆ f64                    ┆ i32         │
╞═════════════╪════════════════════════╪═════════════╡
│ 7912392     ┆ 1.0                    ┆ 0           │
│ 7396997     ┆ 1.0                    ┆ 0           │
│ 690290      ┆ 1.0                    ┆ 0           │
│ 6323569     ┆ 1.0                    ┆ 0           │
│ 3393660     ┆ 1.0                    ┆ 0           │
│ 7006636     ┆ 1.0                    ┆ 0           │
│ 6911461     ┆ 1.166667               ┆ 0           │
│ 6500923     ┆ 1.0                    ┆ 0           │
│ 2230077     ┆ 1.0                    ┆ 0           │
│ 7680497     ┆ 1.0                    ┆ 0           │
└─────────────┴────────────────────────┴─────────────┘

--- Ví dụ khách hàng thuộc segment 1 ---
shape: (10, 3)
┌─────

In [317]:
segment_counts = (
    customer_final
    .group_by("buy_segment")
    .agg(pl.count().alias("num_customers"))
    .sort("buy_segment")
)

print("=== Số lượng khách hàng theo từng segment (0=ít, 1=vừa, 2=nhiều) ===")
print(segment_counts)


=== Số lượng khách hàng theo từng segment (0=ít, 1=vừa, 2=nhiều) ===
shape: (3, 2)
┌─────────────┬───────────────┐
│ buy_segment ┆ num_customers │
│ ---         ┆ ---           │
│ i32         ┆ u32           │
╞═════════════╪═══════════════╡
│ 0           ┆ 1488669       │
│ 1           ┆ 818010        │
│ 2           ┆ 135627        │
└─────────────┴───────────────┘


C:\Users\PC\AppData\Local\Temp\ipykernel_12820\670758724.py:4: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  .agg(pl.count().alias("num_customers"))


In [318]:
# Đặt đường dẫn xuất file
output_path = r"D:\003. HK1 - Năm 3\02. CS116 - Python cho Máy học\CS116-DoAn\Phase-2\Feature engineering\customer_behavior.parquet"

# Tạo thư mục nếu chưa tồn tại
os.makedirs(os.path.dirname(output_path), exist_ok=True)

# Lưu dataframe
customer_final.write_parquet(output_path)

print("Đã lưu file thành công tại:")
print(output_path)

Đã lưu file thành công tại:
D:\003. HK1 - Năm 3\02. CS116 - Python cho Máy học\CS116-DoAn\Phase-2\Feature engineering\customer_behavior.parquet


**Biến chính là buy_segment** còn lại là biến trung gian

### **Đặc trưng:** Mức độ cao cấp của khách hàng - dựa trên "Sữa", "Tã"
- Bước 1: Gom nhóm giá các mặt hàng có category_l1 = 'sữa' theo các nhãn "Bình dân", "Trung cấp" và "cao cấp".

- Bước 2: Thống kê xem khách hàng đó mua sữa thuộc nhóm nào nhiều nhất để gán nhãn cho khách hàng đó.

In [319]:

# Đường dẫn item file đã preprocessing
path_item = r"D:\003. HK1 - Năm 3\02. CS116 - Python cho Máy học\CS116-DoAn\Phase-2\Feature engineering\sale_pers.item_chunk_0.parquet"

# Chỉ load cột cần thiết
df_item = pl.read_parquet(
    path_item,
    columns=["item_id", "category_l1", "price_segment"]
)

print("ITEM shape:", df_item.shape)
print(df_item.head())


ITEM shape: (27323, 3)
shape: (5, 3)
┌───────────────┬────────────────┬───────────────┐
│ item_id       ┆ category_l1    ┆ price_segment │
│ ---           ┆ ---            ┆ ---           │
│ str           ┆ str            ┆ i64           │
╞═══════════════╪════════════════╪═══════════════╡
│ 0502020000004 ┆ Babycare       ┆ 0             │
│ 0010290040150 ┆ Thời trang     ┆ 0             │
│ 0008010000015 ┆ Đồ chơi & Sách ┆ 0             │
│ 0020010000094 ┆ Tã             ┆ 1             │
│ 0020010000098 ┆ Tã             ┆ 1             │
└───────────────┴────────────────┴───────────────┘


In [320]:
# Nơi lưu nhiều parquet transaction chunk
trans_folder = r"D:\003. HK1 - Năm 3\02. CS116 - Python cho Máy học\CS116-DoAn\Phase-2\preprocessing data"

transaction_chunks = []
for i in range(20):   # vì có chunk_0 đến chunk_19
    path_t = fr"{trans_folder}\sale_pers.purchase_history_daily_chunk_{i}.parquet"
    print("Loading:", path_t)

    df_t = pl.read_parquet(
        path_t,
        columns=["customer_id", "item_id", "created_date"]  # chỉ load cột cần thiết
    )

    transaction_chunks.append(df_t)

print("Tải xong tất cả transaction chunks")


Loading: D:\003. HK1 - Năm 3\02. CS116 - Python cho Máy học\CS116-DoAn\Phase-2\preprocessing data\sale_pers.purchase_history_daily_chunk_0.parquet
Loading: D:\003. HK1 - Năm 3\02. CS116 - Python cho Máy học\CS116-DoAn\Phase-2\preprocessing data\sale_pers.purchase_history_daily_chunk_1.parquet
Loading: D:\003. HK1 - Năm 3\02. CS116 - Python cho Máy học\CS116-DoAn\Phase-2\preprocessing data\sale_pers.purchase_history_daily_chunk_2.parquet
Loading: D:\003. HK1 - Năm 3\02. CS116 - Python cho Máy học\CS116-DoAn\Phase-2\preprocessing data\sale_pers.purchase_history_daily_chunk_3.parquet
Loading: D:\003. HK1 - Năm 3\02. CS116 - Python cho Máy học\CS116-DoAn\Phase-2\preprocessing data\sale_pers.purchase_history_daily_chunk_4.parquet
Loading: D:\003. HK1 - Năm 3\02. CS116 - Python cho Máy học\CS116-DoAn\Phase-2\preprocessing data\sale_pers.purchase_history_daily_chunk_5.parquet
Loading: D:\003. HK1 - Năm 3\02. CS116 - Python cho Máy học\CS116-DoAn\Phase-2\preprocessing data\sale_pers.purchase_h

In [321]:
df_trans = pl.concat(transaction_chunks, how="vertical")
print("TRANSACTION shape:", df_trans.shape)
print(df_trans.head())


TRANSACTION shape: (35729825, 3)
shape: (5, 3)
┌─────────────┬───────────────┬──────────────┐
│ customer_id ┆ item_id       ┆ created_date │
│ ---         ┆ ---           ┆ ---          │
│ i32         ┆ str           ┆ date         │
╞═════════════╪═══════════════╪══════════════╡
│ 5254214     ┆ 7115000000004 ┆ 2024-12-24   │
│ 7573232     ┆ 0029130000030 ┆ 2024-12-24   │
│ 8187418     ┆ 3496000000053 ┆ 2024-12-24   │
│ 8187418     ┆ 2700000000002 ┆ 2024-12-24   │
│ 6931560     ┆ 0029110000036 ┆ 2024-12-28   │
└─────────────┴───────────────┴──────────────┘


In [322]:
df_join = df_trans.join(
    df_item,
    on="item_id",
    how="left"
)

print("JOIN shape:", df_join.shape)
print(df_join.head())


JOIN shape: (35729825, 5)
shape: (5, 5)
┌─────────────┬───────────────┬──────────────┬──────────────────┬───────────────┐
│ customer_id ┆ item_id       ┆ created_date ┆ category_l1      ┆ price_segment │
│ ---         ┆ ---           ┆ ---          ┆ ---              ┆ ---           │
│ i32         ┆ str           ┆ date         ┆ str              ┆ i64           │
╞═════════════╪═══════════════╪══════════════╪══════════════════╪═══════════════╡
│ 5254214     ┆ 7115000000004 ┆ 2024-12-24   ┆ Thực phẩm cho bé ┆ 0             │
│ 7573232     ┆ 0029130000030 ┆ 2024-12-24   ┆ Thực phẩm cho bé ┆ 0             │
│ 8187418     ┆ 3496000000053 ┆ 2024-12-24   ┆ Thời trang       ┆ 0             │
│ 8187418     ┆ 2700000000002 ┆ 2024-12-24   ┆ Vệ sinh          ┆ 1             │
│ 6931560     ┆ 0029110000036 ┆ 2024-12-28   ┆ Thực phẩm cho bé ┆ 1             │
└─────────────┴───────────────┴──────────────┴──────────────────┴───────────────┘


In [323]:
important_cats = ["Sữa", "Tã"]

df_filtered = df_join.filter(
    pl.col("category_l1").is_in(important_cats)
)

print("Dữ liệu sau khi lọc 3 nhóm quan trọng:", df_filtered.shape)
print(df_filtered.head())


Dữ liệu sau khi lọc 3 nhóm quan trọng: (7661159, 5)
shape: (5, 5)
┌─────────────┬───────────────┬──────────────┬─────────────┬───────────────┐
│ customer_id ┆ item_id       ┆ created_date ┆ category_l1 ┆ price_segment │
│ ---         ┆ ---           ┆ ---          ┆ ---         ┆ ---           │
│ i32         ┆ str           ┆ date         ┆ str         ┆ i64           │
╞═════════════╪═══════════════╪══════════════╪═════════════╪═══════════════╡
│ 3353278     ┆ 2242000910001 ┆ 2024-12-24   ┆ Tã          ┆ 1             │
│ 7573978     ┆ 0020010000438 ┆ 2024-12-24   ┆ Sữa         ┆ 1             │
│ 4810993     ┆ 2263000000021 ┆ 2024-12-28   ┆ Tã          ┆ 0             │
│ 7901749     ┆ 3773000000004 ┆ 2024-12-24   ┆ Sữa         ┆ 2             │
│ 7555248     ┆ 3773000000004 ┆ 2024-12-24   ┆ Sữa         ┆ 2             │
└─────────────┴───────────────┴──────────────┴─────────────┴───────────────┘


In [324]:
customer_segment_count = (
    df_filtered
    .group_by(["customer_id", "price_segment"])
    .agg(pl.count().alias("purchase_count"))
)

print("=== Mẫu thống kê số lần mua theo phân khúc giá ===")
print(customer_segment_count.head(10))


C:\Users\PC\AppData\Local\Temp\ipykernel_12820\3378104036.py:4: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  .agg(pl.count().alias("purchase_count"))


=== Mẫu thống kê số lần mua theo phân khúc giá ===
shape: (10, 3)
┌─────────────┬───────────────┬────────────────┐
│ customer_id ┆ price_segment ┆ purchase_count │
│ ---         ┆ ---           ┆ ---            │
│ i32         ┆ i64           ┆ u32            │
╞═════════════╪═══════════════╪════════════════╡
│ 3915079     ┆ 1             ┆ 1              │
│ 335117      ┆ 0             ┆ 9              │
│ 703093      ┆ 0             ┆ 5              │
│ 4200016     ┆ 1             ┆ 5              │
│ 2524669     ┆ 2             ┆ 1              │
│ 5563639     ┆ 1             ┆ 5              │
│ 7232654     ┆ 0             ┆ 1              │
│ 4297645     ┆ 0             ┆ 4              │
│ 7570923     ┆ 1             ┆ 1              │
│ 7960483     ┆ 0             ┆ 8              │
└─────────────┴───────────────┴────────────────┘


In [325]:
customer_pivot = (
    customer_segment_count
    .pivot(
        index="customer_id",
        columns="price_segment",
        values="purchase_count",
        aggregate_function="first",
    )
    .fill_null(0)
    .rename({"0": "seg_0", "1": "seg_1", "2": "seg_2"})
)

print("=== Pivot bảng phân khúc giá theo customer ===")
print(customer_pivot.head(10))


C:\Users\PC\AppData\Local\Temp\ipykernel_12820\2920628322.py:3: DeprecationWarning: the argument `columns` for `DataFrame.pivot` is deprecated. It was renamed to `on` in version 1.0.0.
  .pivot(


=== Pivot bảng phân khúc giá theo customer ===
shape: (10, 4)
┌─────────────┬───────┬───────┬───────┐
│ customer_id ┆ seg_1 ┆ seg_0 ┆ seg_2 │
│ ---         ┆ ---   ┆ ---   ┆ ---   │
│ i32         ┆ u32   ┆ u32   ┆ u32   │
╞═════════════╪═══════╪═══════╪═══════╡
│ 3915079     ┆ 1     ┆ 2     ┆ 0     │
│ 335117      ┆ 0     ┆ 9     ┆ 0     │
│ 703093      ┆ 25    ┆ 5     ┆ 0     │
│ 4200016     ┆ 5     ┆ 1     ┆ 0     │
│ 2524669     ┆ 33    ┆ 1     ┆ 1     │
│ 5563639     ┆ 5     ┆ 0     ┆ 0     │
│ 7232654     ┆ 3     ┆ 1     ┆ 11    │
│ 4297645     ┆ 2     ┆ 4     ┆ 0     │
│ 7570923     ┆ 1     ┆ 0     ┆ 0     │
│ 7960483     ┆ 0     ┆ 8     ┆ 0     │
└─────────────┴───────┴───────┴───────┘


In [326]:
num_customers = customer_pivot.select(pl.count()).item()
print("Số lượng khách hàng:", num_customers)


Số lượng khách hàng: 1276677


C:\Users\PC\AppData\Local\Temp\ipykernel_12820\1016690670.py:1: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  num_customers = customer_pivot.select(pl.count()).item()


In [327]:
customer_luxury = customer_pivot.with_columns([
    # max segment theo logic: ưu tiên phân khúc cao hơn nếu tie
    pl.when((pl.col("seg_2") >= pl.col("seg_1")) & (pl.col("seg_2") >= pl.col("seg_0")))
        .then(2)
    .when((pl.col("seg_1") >= pl.col("seg_0")) & (pl.col("seg_1") >= pl.col("seg_2")))
        .then(1)
    .otherwise(0)
    .alias("luxury_level")
])

print("=== Mẫu luxury level của customer ===")
print(customer_luxury.head(10))


=== Mẫu luxury level của customer ===
shape: (10, 5)
┌─────────────┬───────┬───────┬───────┬──────────────┐
│ customer_id ┆ seg_1 ┆ seg_0 ┆ seg_2 ┆ luxury_level │
│ ---         ┆ ---   ┆ ---   ┆ ---   ┆ ---          │
│ i32         ┆ u32   ┆ u32   ┆ u32   ┆ i32          │
╞═════════════╪═══════╪═══════╪═══════╪══════════════╡
│ 3915079     ┆ 1     ┆ 2     ┆ 0     ┆ 0            │
│ 335117      ┆ 0     ┆ 9     ┆ 0     ┆ 0            │
│ 703093      ┆ 25    ┆ 5     ┆ 0     ┆ 1            │
│ 4200016     ┆ 5     ┆ 1     ┆ 0     ┆ 1            │
│ 2524669     ┆ 33    ┆ 1     ┆ 1     ┆ 1            │
│ 5563639     ┆ 5     ┆ 0     ┆ 0     ┆ 1            │
│ 7232654     ┆ 3     ┆ 1     ┆ 11    ┆ 2            │
│ 4297645     ┆ 2     ┆ 4     ┆ 0     ┆ 0            │
│ 7570923     ┆ 1     ┆ 0     ┆ 0     ┆ 1            │
│ 7960483     ┆ 0     ┆ 8     ┆ 0     ┆ 0            │
└─────────────┴───────┴───────┴───────┴──────────────┘


In [328]:
level_counts = (
    customer_luxury
    .group_by("luxury_level")
    .agg(pl.count().alias("num_customers"))
    .sort("luxury_level")
)

print("\n=== Số lượng khách hàng theo từng luxury_level ===")
print(level_counts)



=== Số lượng khách hàng theo từng luxury_level ===
shape: (3, 2)
┌──────────────┬───────────────┐
│ luxury_level ┆ num_customers │
│ ---          ┆ ---           │
│ i32          ┆ u32           │
╞══════════════╪═══════════════╡
│ 0            ┆ 399426        │
│ 1            ┆ 784154        │
│ 2            ┆ 93097         │
└──────────────┴───────────────┘


C:\Users\PC\AppData\Local\Temp\ipykernel_12820\3369559557.py:4: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  .agg(pl.count().alias("num_customers"))


In [329]:
output_path = r"D:\003. HK1 - Năm 3\02. CS116 - Python cho Máy học\CS116-DoAn\Phase-2\Feature engineering\customer_luxury.parquet"

# Tạo thư mục nếu chưa tồn tại
os.makedirs(os.path.dirname(output_path), exist_ok=True)

# Lưu file
customer_luxury.write_parquet(output_path)

print("Đã lưu file thành công tại:")
print(output_path)

Đã lưu file thành công tại:
D:\003. HK1 - Năm 3\02. CS116 - Python cho Máy học\CS116-DoAn\Phase-2\Feature engineering\customer_luxury.parquet


### **Đặc trưng:** Tuổi hiện tại của bé
Dựa trên age_group, step_1, step_2 và mom

In [31]:
import glob
import re
from datetime import date

In [32]:
ITEM_PATH = r"D:\003. HK1 - Năm 3\02. CS116 - Python cho Máy học\CS116-DoAn\Phase-2\Feature engineering\sale_pers.item_chunk_0.parquet"
TRANS_DIR =  r"D:\003. HK1 - Năm 3\02. CS116 - Python cho Máy học\CS116-DoAn\Phase-2\preprocessing data"

OUTPUT_PATH = r"D:\003. HK1 - Năm 3\02. CS116 - Python cho Máy học\CS116-DoAn\Phase-2\Feature engineering\customer_age_features.parquet"

PREDICTION_DATE = date(2025, 1, 1)

STEP1_AGE_MONTHS = 3.0       # Step1 dùng cho 0–6M → trung bình ~3M
AGE_0_3M_MONTHS = 1.5        # 0–3M → trung bình ~1.5M
MOM_AGE_MONTHS = 0.0         # bạn yêu cầu = 0


In [33]:
print("Đang load ITEM...")
item = pl.read_parquet(ITEM_PATH)

item = item.select([
    "item_id", 
    "category_l1", "category_l2",
    "description_final",
    "age_group_final"
])

item = item.with_columns(
    pl.col("description_final").str.to_lowercase().alias("desc_lc")
)



Đang load ITEM...


In [34]:
# FLAG Step1 (chỉ nhận khi category_l1 = Sữa)

step1_positive = [
    "step 1", "step-1", "step1",
    "stage 1", "stage-1", "stage1",
    "cho bé 0+", "cho bé 0-6", "từ 0 tháng",
    "trẻ sơ sinh", "newborn"
]

step1_negative = [
    "bước 1:"       # chắc chắn là hướng dẫn
]


item = item.with_columns([

    # Positive: chứa 1 trong các pattern
    pl.any_horizontal([
        pl.col("desc_lc").str.contains(pat)
        for pat in step1_positive
    ]).alias("tmp_step1_pos"),

    # Negative
    pl.any_horizontal([
        pl.col("desc_lc").str.contains(pat)
        for pat in step1_negative
    ]).alias("tmp_step1_neg"),
])

item = item.with_columns([
    (
        (pl.col("category_l1") == "Sữa") &
        pl.col("tmp_step1_pos") &
        (~pl.col("tmp_step1_neg"))
    ).alias("is_step1")
]).drop(["tmp_step1_pos", "tmp_step1_neg"])



In [35]:
item.select(
    pl.col("is_step1").value_counts()
)


is_step1
struct[2]
"{true,62}"
"{false,27261}"


In [37]:
# ====== FLAG is_age_0_6M (CHUẨN THEO ĐỊNH NGHĨA CUỐI) ======

item = item.with_columns(
    pl.col("age_group_final")
    .cast(pl.Utf8)
    .str.to_lowercase()
    .alias("age_lc")
)

# (A) RANGE giao [0,6]: min < 6, max <= 6
# Bắt các dạng: 0-3M, 1-5M, 2-6M, 3-6M, 4-6M, 5-6M
regex_range_0_6 = r"\b[0-5]\s*[-–]\s*[1-6]\s*m\b"

# (B) Dạng "Từ xM" với x ∈ {0,1,2,3}
regex_from_0_3 = r"\btừ\s*[0-3]\s*m\b"

item = item.with_columns(
    (
        pl.col("age_lc").str.contains(regex_range_0_6) |
        pl.col("age_lc").str.contains(regex_from_0_3)
    ).alias("is_age_0_3M")
)


In [38]:
item.select(
    pl.col("is_age_0_3M").value_counts()
)


is_age_0_3M
struct[2]
"{false,25839}"
"{true,1484}"


In [39]:
#FLAG is_mom (cẩn thận ĐẦM BẦU)

mom_positive = [
    #"sữa bầu",
    #"sua bau",
    #"sữa cho mẹ",
    #"dinh dưỡng cho mẹ",
    #"dành cho mẹ",
    "mom"
]


mom_negative = [
    "đầm bầu", "váy bầu", "áo bầu"
]

item = item.with_columns([
    pl.any_horizontal([
        pl.col("desc_lc").str.contains(pat)
        for pat in mom_positive
    ]).alias("tmp_mom_pos"),

    pl.any_horizontal([
        pl.col("desc_lc").str.contains(pat)
        for pat in mom_negative
    ]).alias("tmp_mom_neg"),
])

item = item.with_columns([
    (
        (pl.col("tmp_mom_pos")) & 
        (~pl.col("tmp_mom_neg"))
    ).alias("is_mom")
]).drop(["tmp_mom_pos", "tmp_mom_neg"])


In [40]:
#ĐỌC 20 CHUNK TRANSACTION + JOIN ITEM + TÍNH NGÀY MIN/MAX

transaction_files = sorted(glob.glob(TRANS_DIR + r"\sale_pers.purchase_history_daily_chunk_*.parquet"))
print("Số file transaction:", len(transaction_files))

per_chunk_stats = []


Số file transaction: 20


In [41]:
for file in transaction_files:
    print("Đang xử lý:", file)

    trans = (
        pl.read_parquet(file)
        .select(["item_id", "customer_id", "created_date"])
        .with_columns(pl.col("created_date").cast(pl.Date))
    )

    # JOIN ITEM
    trans_j = trans.join(
        item.select(["item_id", "is_step1", "is_age_0_3M", "is_mom"]),
        on="item_id",
        how="left"
    )

    # AGG theo customer
    stats = (
        trans_j.group_by("customer_id")
        .agg([
            pl.col("created_date").filter(pl.col("is_step1")).min().alias("first_date_buy_step1"),
            pl.col("created_date").filter(pl.col("is_age_0_3M")).min().alias("first_date_buy_age_group_0_3M"),
            pl.col("created_date").filter(pl.col("is_mom")).max().alias("last_date_buy_milk4mom"),
        ])
    )

    per_chunk_stats.append(stats)


Đang xử lý: D:\003. HK1 - Năm 3\02. CS116 - Python cho Máy học\CS116-DoAn\Phase-2\preprocessing data\sale_pers.purchase_history_daily_chunk_0.parquet
Đang xử lý: D:\003. HK1 - Năm 3\02. CS116 - Python cho Máy học\CS116-DoAn\Phase-2\preprocessing data\sale_pers.purchase_history_daily_chunk_1.parquet
Đang xử lý: D:\003. HK1 - Năm 3\02. CS116 - Python cho Máy học\CS116-DoAn\Phase-2\preprocessing data\sale_pers.purchase_history_daily_chunk_10.parquet
Đang xử lý: D:\003. HK1 - Năm 3\02. CS116 - Python cho Máy học\CS116-DoAn\Phase-2\preprocessing data\sale_pers.purchase_history_daily_chunk_11.parquet
Đang xử lý: D:\003. HK1 - Năm 3\02. CS116 - Python cho Máy học\CS116-DoAn\Phase-2\preprocessing data\sale_pers.purchase_history_daily_chunk_12.parquet
Đang xử lý: D:\003. HK1 - Năm 3\02. CS116 - Python cho Máy học\CS116-DoAn\Phase-2\preprocessing data\sale_pers.purchase_history_daily_chunk_13.parquet
Đang xử lý: D:\003. HK1 - Năm 3\02. CS116 - Python cho Máy học\CS116-DoAn\Phase-2\preprocessing 

In [42]:
#GỘP KẾT QUẢ CHUNK + LẤY MIN/MAX CUỐI CÙNG

print("\n=== Gộp tất cả chunk lại ===")
customer_dates_all = pl.concat(per_chunk_stats, how="vertical_relaxed")
print("Shape sau gộp:", customer_dates_all.shape)


customer_dates_final = (
    customer_dates_all
    .group_by("customer_id")
    .agg([
        pl.col("first_date_buy_step1").min().alias("first_date_buy_step1"),
        pl.col("first_date_buy_age_group_0_3M").min().alias("first_date_buy_age_group_0_3M"),
        pl.col("last_date_buy_milk4mom").max().alias("last_date_buy_milk4mom"),
    ])
)



=== Gộp tất cả chunk lại ===
Shape sau gộp: (9875793, 4)


In [43]:
print("\n=== 20 dòng mẫu có thông tin tuổi ===")
sample_nonnull = customer_dates_final.filter(
    pl.col("first_date_buy_step1").is_not_null() |
    pl.col("first_date_buy_age_group_0_3M").is_not_null() |
    pl.col("last_date_buy_milk4mom").is_not_null()
).head(20)

print(sample_nonnull)



=== 20 dòng mẫu có thông tin tuổi ===
shape: (20, 4)
┌─────────────┬──────────────────────┬───────────────────────────────┬────────────────────────┐
│ customer_id ┆ first_date_buy_step1 ┆ first_date_buy_age_group_0_3M ┆ last_date_buy_milk4mom │
│ ---         ┆ ---                  ┆ ---                           ┆ ---                    │
│ i32         ┆ date                 ┆ date                          ┆ date                   │
╞═════════════╪══════════════════════╪═══════════════════════════════╪════════════════════════╡
│ 2492224     ┆ null                 ┆ 2024-01-30                    ┆ null                   │
│ 1444127     ┆ null                 ┆ 2024-10-24                    ┆ null                   │
│ 7948706     ┆ null                 ┆ 2024-10-11                    ┆ null                   │
│ 6021827     ┆ null                 ┆ 2024-09-17                    ┆ 2024-11-15             │
│ 6509825     ┆ null                 ┆ 2024-02-07                    ┆ 2024-05-21 

In [44]:
# TÍNH TUỔI HIỆN TẠI (age_by_*)
pred_date_lit = pl.lit(PREDICTION_DATE).cast(pl.Date)

customer_with_age = (
    customer_dates_final
    .with_columns([
        # Lấy số ngày từ hiệu hai ngày
        (pred_date_lit - pl.col("first_date_buy_step1")).dt.total_days().alias("days_from_step1"),
        (pred_date_lit - pl.col("first_date_buy_age_group_0_3M")).dt.total_days().alias("days_from_age_group"),
        (pred_date_lit - pl.col("last_date_buy_milk4mom")).dt.total_days().alias("days_from_milk4mom"),
    ])
    .with_columns([
        (pl.col("days_from_step1") / 30).alias("months_from_step1"),
        (pl.col("days_from_age_group") / 30).alias("months_from_age_group"),
        (pl.col("days_from_milk4mom") / 30).alias("months_from_milk4mom"),
    ])
    .with_columns([
        # FINAL AGE
        (STEP1_AGE_MONTHS + pl.col("months_from_step1")).alias("age_by_step1"),
        (AGE_0_3M_MONTHS + pl.col("months_from_age_group")).alias("age_by_age_group"),
        (MOM_AGE_MONTHS + pl.col("months_from_milk4mom")).alias("age_by_milk4mom"),
    ])
    .select([
        "customer_id",
        "first_date_buy_step1", "age_by_step1",
        "first_date_buy_age_group_0_3M", "age_by_age_group",
        "last_date_buy_milk4mom", "age_by_milk4mom",
    ])
)


In [45]:
# check

print("\n=== THỐNG KÊ NON-NULL ===")

print("Có Step1:", customer_with_age.filter(pl.col("age_by_step1").is_not_null()).height)
print("Có age_group 0–3M:", customer_with_age.filter(pl.col("age_by_age_group").is_not_null()).height)
print("Có Mom:", customer_with_age.filter(pl.col("age_by_milk4mom").is_not_null()).height)



=== THỐNG KÊ NON-NULL ===
Có Step1: 318575
Có age_group 0–3M: 893940
Có Mom: 164508


In [46]:
# check

print("\n=== Mẫu khách có cả 3 tín hiệu ===")
print(
    customer_with_age
    .filter(
        pl.col("age_by_step1").is_not_null() &
        pl.col("age_by_age_group").is_not_null() &
        pl.col("age_by_milk4mom").is_not_null()
    )
    .head(20)
)



=== Mẫu khách có cả 3 tín hiệu ===
shape: (20, 7)
┌─────────────┬──────────────┬─────────────┬─────────────┬─────────────┬─────────────┬─────────────┐
│ customer_id ┆ first_date_b ┆ age_by_step ┆ first_date_ ┆ age_by_age_ ┆ last_date_b ┆ age_by_milk │
│ ---         ┆ uy_step1     ┆ 1           ┆ buy_age_gro ┆ group       ┆ uy_milk4mom ┆ 4mom        │
│ i32         ┆ ---          ┆ ---         ┆ up_0_3M     ┆ ---         ┆ ---         ┆ ---         │
│             ┆ date         ┆ f64         ┆ ---         ┆ f64         ┆ date        ┆ f64         │
│             ┆              ┆             ┆ date        ┆             ┆             ┆             │
╞═════════════╪══════════════╪═════════════╪═════════════╪═════════════╪═════════════╪═════════════╡
│ 8182274     ┆ 2024-12-22   ┆ 3.333333    ┆ 2024-12-22  ┆ 1.833333    ┆ 2024-12-22  ┆ 0.333333    │
│ 4480575     ┆ 2024-04-28   ┆ 11.266667   ┆ 2024-03-11  ┆ 11.366667   ┆ 2024-03-11  ┆ 9.866667    │
│ 6507482     ┆ 2024-06-06   ┆ 9.966667 

In [47]:
print("\n=== Random 20 khách có cả 3 tín hiệu ===")
print(
    customer_with_age
    .filter(
        pl.col("age_by_step1").is_not_null() &
        pl.col("age_by_age_group").is_not_null() &
        pl.col("age_by_milk4mom").is_not_null()
    )
    .sample(n=20, with_replacement=False)
)



=== Random 20 khách có cả 3 tín hiệu ===
shape: (20, 7)
┌─────────────┬──────────────┬─────────────┬─────────────┬─────────────┬─────────────┬─────────────┐
│ customer_id ┆ first_date_b ┆ age_by_step ┆ first_date_ ┆ age_by_age_ ┆ last_date_b ┆ age_by_milk │
│ ---         ┆ uy_step1     ┆ 1           ┆ buy_age_gro ┆ group       ┆ uy_milk4mom ┆ 4mom        │
│ i32         ┆ ---          ┆ ---         ┆ up_0_3M     ┆ ---         ┆ ---         ┆ ---         │
│             ┆ date         ┆ f64         ┆ ---         ┆ f64         ┆ date        ┆ f64         │
│             ┆              ┆             ┆ date        ┆             ┆             ┆             │
╞═════════════╪══════════════╪═════════════╪═════════════╪═════════════╪═════════════╪═════════════╡
│ 7077592     ┆ 2024-01-02   ┆ 15.166667   ┆ 2024-01-28  ┆ 12.8        ┆ 2024-05-19  ┆ 7.566667    │
│ 3659280     ┆ 2024-09-17   ┆ 6.533333    ┆ 2024-01-15  ┆ 13.233333   ┆ 2024-10-10  ┆ 2.766667    │
│ 7450495     ┆ 2024-05-23   ┆ 10.

 **age_by_milk4mom không tốt lắm nên không dùng nó để tạo cột tuổi trung bình cuối cùng.**

In [48]:
total_customers = customer_with_age.select(
    pl.col("customer_id").n_unique()
).item()

print("Tổng số customer:", total_customers)


Tổng số customer: 2442306


In [49]:
customers_with_baby_signal = (
    customer_with_age
    .filter(
        pl.col("age_by_step1").is_not_null() |
        pl.col("age_by_age_group").is_not_null()
    )
    .select(pl.col("customer_id").n_unique())
    .item()
)

print("Customer có ít nhất 1 tín hiệu em bé:", customers_with_baby_signal)


Customer có ít nhất 1 tín hiệu em bé: 958375


Chỉ xét có cả 2:
- <=3 tháng  : 222,030  (~87.4%)
- 3–6 tháng : 22,130   (~8.7%)
- >6 tháng  : 9,980    (~3.9%)


In [50]:
# tìm ra những khách hàng có cả age_by_step1 và age_by_age_group, tính độ lệch

print(
    customer_with_age
    .filter(
        pl.col("age_by_step1").is_not_null() &
        pl.col("age_by_age_group").is_not_null()
    )
    .select(
        (pl.col("age_by_step1") - pl.col("age_by_age_group")).abs().alias("diff")
    )
    .describe()
)


shape: (9, 2)
┌────────────┬──────────┐
│ statistic  ┆ diff     │
│ ---        ┆ ---      │
│ str        ┆ f64      │
╞════════════╪══════════╡
│ count      ┆ 254140.0 │
│ null_count ┆ 0.0      │
│ mean       ┆ 1.888039 │
│ std        ┆ 1.601602 │
│ min        ┆ 0.0      │
│ 25%        ┆ 1.466667 │
│ 50%        ┆ 1.5      │
│ 75%        ┆ 1.533333 │
│ max        ┆ 13.6     │
└────────────┴──────────┘


- Trung bình thì lệch tầm 1.7 tháng (mean = 1.67 (tháng))
- Có outlier khá mạnh (max = 10.67 tháng)

In [51]:
# Các khách hàng có độ lệch lớn nhất
large_diff = (
    customer_with_age
    .filter(
        pl.col("age_by_step1").is_not_null() &
        pl.col("age_by_age_group").is_not_null()
    )
    .with_columns(
        (pl.col("age_by_step1") - pl.col("age_by_age_group"))
        .abs()
        .alias("diff")
    )
    .sort("diff", descending=True)
)

print("\n=== TOP 20 khách hàng có độ lệch tuổi lớn nhất ===")
print(
    large_diff.select([
        "customer_id",
        "first_date_buy_step1",
        "age_by_step1",
        "first_date_buy_age_group_0_3M",
        "age_by_age_group",
        "diff"
    ]).head(20)
)



=== TOP 20 khách hàng có độ lệch tuổi lớn nhất ===
shape: (20, 6)
┌─────────────┬───────────────────┬──────────────┬──────────────────┬──────────────────┬───────────┐
│ customer_id ┆ first_date_buy_st ┆ age_by_step1 ┆ first_date_buy_a ┆ age_by_age_group ┆ diff      │
│ ---         ┆ ep1               ┆ ---          ┆ ge_group_0_3M    ┆ ---              ┆ ---       │
│ i32         ┆ ---               ┆ f64          ┆ ---              ┆ f64              ┆ f64       │
│             ┆ date              ┆              ┆ date             ┆                  ┆           │
╞═════════════╪═══════════════════╪══════════════╪══════════════════╪══════════════════╪═══════════╡
│ 2099179     ┆ 2024-01-01        ┆ 15.2         ┆ 2024-12-29       ┆ 1.6              ┆ 13.6      │
│ 2058087     ┆ 2024-01-02        ┆ 15.166667    ┆ 2024-12-29       ┆ 1.6              ┆ 13.566667 │
│ 6871092     ┆ 2024-01-01        ┆ 15.2         ┆ 2024-12-27       ┆ 1.666667         ┆ 13.533333 │
│ 5120249     ┆ 2024-01-

In [52]:
# Đếm số khách hàng có độ lệch > 6 tháng
num_diff_gt_10 = (
    large_diff
    .filter(pl.col("diff") > 5)
    .select(pl.count())
    .item()
)

print(f"\n=== Số khách hàng có |age_by_step1 - age_by_age_group| > 5 tháng ===")
print(num_diff_gt_10)



=== Số khách hàng có |age_by_step1 - age_by_age_group| > 5 tháng ===
14996


C:\Users\PC\AppData\Local\Temp\ipykernel_20440\578139962.py:5: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  .select(pl.count())


In [53]:
customer_with_age = customer_with_age.with_columns([
    # Tính độ lệch nếu có đủ 2 tín hiệu
    pl.when(
        pl.col("age_by_step1").is_not_null() &
        pl.col("age_by_age_group").is_not_null()
    )
    .then((pl.col("age_by_step1") - pl.col("age_by_age_group")).abs())
    .otherwise(None)
    .alias("age_diff")
])

customer_with_age = customer_with_age.with_columns([
    pl.when(
        # chỉ có step1
        pl.col("age_by_step1").is_not_null() &
        pl.col("age_by_age_group").is_null()
    ).then(pl.col("age_by_step1"))

    .when(
        # chỉ có age_group
        pl.col("age_by_step1").is_null() &
        pl.col("age_by_age_group").is_not_null()
    ).then(pl.col("age_by_age_group"))

    .when(
        # lệch <= 3 tháng → trung bình
        pl.col("age_diff") <= 3
    ).then(
        (pl.col("age_by_step1") + pl.col("age_by_age_group")) / 2
    )

    .when(
        # 3 < lệch <= 6 → ưu tiên step1
        (pl.col("age_diff") > 3) & (pl.col("age_diff") <= 5)
    ).then(pl.col("age_by_step1"))

    .otherwise(None)   # lệch > 6 → loại
    .alias("age_final")
])


In [55]:
customer_with_age.head(10)


customer_id,first_date_buy_step1,age_by_step1,first_date_buy_age_group_0_3M,age_by_age_group,last_date_buy_milk4mom,age_by_milk4mom,age_diff,age_final
i32,date,f64,date,f64,date,f64,f64,f64
2492224,null,null,2024-01-30,12.733333,null,null,null,12.733333
7366843,null,null,null,null,null,null,null,null
1867259,null,null,null,null,null,null,null,null
1444127,null,null,2024-10-24,3.8,null,null,null,3.8
7668445,null,null,null,null,null,null,null,null
3416758,null,null,null,null,null,null,null,null
7829299,null,null,null,null,null,null,null,null
5050782,null,null,null,null,null,null,null,null
7405101,null,null,null,null,null,null,null,null


In [58]:
diff_bucket_stats = (
    customer_with_age
    .filter(
        pl.col("age_by_step1").is_not_null() &
        pl.col("age_by_age_group").is_not_null()
    )
    .with_columns(
        pl.when(pl.col("age_diff") <= 3)
          .then(pl.lit("<=3"))
        .when((pl.col("age_diff") > 3) & (pl.col("age_diff") <= 5))
          .then(pl.lit("3-6"))
        .otherwise(pl.lit(">5"))
        .alias("diff_bucket")
    )
    .group_by("diff_bucket")
    .agg(pl.count().alias("num_customers"))
    .sort("diff_bucket")
)

print("=== Phân bố độ lệch tuổi ===")
print(diff_bucket_stats)


=== Phân bố độ lệch tuổi ===
shape: (3, 2)
┌─────────────┬───────────────┐
│ diff_bucket ┆ num_customers │
│ ---         ┆ ---           │
│ str         ┆ u32           │
╞═════════════╪═══════════════╡
│ 3-6         ┆ 17114         │
│ <=3         ┆ 222030        │
│ >5          ┆ 14996         │
└─────────────┴───────────────┘


C:\Users\PC\AppData\Local\Temp\ipykernel_20440\3055082791.py:16: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  .agg(pl.count().alias("num_customers"))


In [59]:
customer_with_age.write_parquet(OUTPUT_PATH)
print("Đã lưu:", OUTPUT_PATH)


Đã lưu: D:\003. HK1 - Năm 3\02. CS116 - Python cho Máy học\CS116-DoAn\Phase-2\Feature engineering\customer_age_features.parquet


### **Đặc trưng:** Có mua đa dạng brand không. 
Cũng chỉ xét "Sữa", "Tã"

In [358]:
path_item = r"D:\003. HK1 - Năm 3\02. CS116 - Python cho Máy học\CS116-DoAn\Phase-2\Feature engineering\sale_pers.item_chunk_0.parquet"
df_item = pl.read_parquet(path_item)

print(df_item.columns)


['item_id', 'price', 'category_l1', 'category_l2', 'category_l3', 'category', 'item_type', 'gender_target_final', 'description_final', 'brand_final', 'age_group_final', 'price_segment']


In [359]:
# df_trans: transaction (đã load từng chunk rồi concat hoặc xử lý chunk-wise)
# df_item: item đã có category_l1, brand_final

df = (
    df_trans
    .select(["item_id", "customer_id"])
    .join(
        df_item.select(["item_id", "category_l1", "brand_final"]),
        on="item_id",
        how="left"
    )
    .filter(
        pl.col("category_l1").is_in(["Sữa", "Tã"]) &
        pl.col("brand_final").is_not_null()
    )
)

brand_agg = (
    df
    .group_by(["customer_id", "category_l1"])
    .agg([
        pl.count().alias("num_transactions"),
        pl.col("brand_final").n_unique().alias("num_unique_brands"),
    ])
    .with_columns(
        (pl.col("num_unique_brands") / pl.col("num_transactions"))
        .alias("brand_diversity_ratio")
    )
)


C:\Users\PC\AppData\Local\Temp\ipykernel_12820\2008330972.py:22: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  pl.count().alias("num_transactions"),


In [360]:
def cluster_brand_diversity(pdf):
    X = pdf[["brand_diversity_ratio"]].to_numpy()

    kmeans = KMeans(n_clusters=3, random_state=42, n_init="auto")
    labels = kmeans.fit_predict(X)

    pdf["cluster_raw"] = labels

    # Sắp cluster theo mean ratio
    means = (
        pdf.groupby("cluster_raw")["brand_diversity_ratio"]
        .mean()
        .sort_values()
    )

    mapping = {cluster: i for i, cluster in enumerate(means.index)}
    pdf["brand_segment"] = pdf["cluster_raw"].map(mapping)

    return pdf.drop(columns=["cluster_raw"])


In [361]:
# Tách theo category_l1
milk_pdf = brand_agg.filter(pl.col("category_l1") == "Sữa").to_pandas()
diaper_pdf = brand_agg.filter(pl.col("category_l1") == "Tã").to_pandas()

milk_clustered = cluster_brand_diversity(milk_pdf)
diaper_clustered = cluster_brand_diversity(diaper_pdf)

brand_segment_df = pl.concat([
    pl.from_pandas(milk_clustered),
    pl.from_pandas(diaper_clustered)
])


In [362]:
print(
    brand_segment_df
    .group_by(["category_l1", "brand_segment"])
    .agg(pl.count().alias("num_customers"))
    .sort(["category_l1", "brand_segment"])
)


shape: (6, 3)
┌─────────────┬───────────────┬───────────────┐
│ category_l1 ┆ brand_segment ┆ num_customers │
│ ---         ┆ ---           ┆ ---           │
│ str         ┆ i64           ┆ u32           │
╞═════════════╪═══════════════╪═══════════════╡
│ Sữa         ┆ 0             ┆ 227926        │
│ Sữa         ┆ 1             ┆ 233241        │
│ Sữa         ┆ 2             ┆ 435776        │
│ Tã          ┆ 0             ┆ 227775        │
│ Tã          ┆ 1             ┆ 163506        │
│ Tã          ┆ 2             ┆ 375697        │
└─────────────┴───────────────┴───────────────┘


C:\Users\PC\AppData\Local\Temp\ipykernel_12820\3827601304.py:4: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  .agg(pl.count().alias("num_customers"))


In [363]:
print(
    brand_segment_df
    .group_by(["category_l1", "brand_segment"])
    .agg([
        pl.col("brand_diversity_ratio").min().alias("min"),
        pl.col("brand_diversity_ratio").mean().alias("mean"),
        pl.col("brand_diversity_ratio").median().alias("median"),
        pl.col("brand_diversity_ratio").max().alias("max"),
    ])
    .sort(["category_l1", "brand_segment"])
)


shape: (6, 6)
┌─────────────┬───────────────┬──────────┬──────────┬──────────┬──────────┐
│ category_l1 ┆ brand_segment ┆ min      ┆ mean     ┆ median   ┆ max      │
│ ---         ┆ ---           ┆ ---      ┆ ---      ┆ ---      ┆ ---      │
│ str         ┆ i64           ┆ f64      ┆ f64      ┆ f64      ┆ f64      │
╞═════════════╪═══════════════╪══════════╪══════════╪══════════╪══════════╡
│ Sữa         ┆ 0             ┆ 0.003984 ┆ 0.164849 ┆ 0.166667 ┆ 0.315789 │
│ Sữa         ┆ 1             ┆ 0.318182 ┆ 0.470452 ┆ 0.5      ┆ 0.727273 │
│ Sữa         ┆ 2             ┆ 0.733333 ┆ 0.995604 ┆ 1.0      ┆ 1.0      │
│ Tã          ┆ 0             ┆ 0.004184 ┆ 0.209033 ┆ 0.2      ┆ 0.363636 │
│ Tã          ┆ 1             ┆ 0.368421 ┆ 0.524987 ┆ 0.5      ┆ 0.75     │
│ Tã          ┆ 2             ┆ 0.777778 ┆ 0.999263 ┆ 1.0      ┆ 1.0      │
└─────────────┴───────────────┴──────────┴──────────┴──────────┴──────────┘


In [364]:
for seg in [0, 1, 2]:
    print(f"\n=== Random khách segment {seg} ===")
    print(
        brand_segment_df
        .filter(pl.col("brand_segment") == seg)
        .sample(n=10, seed=42)
        .select([
            "customer_id",
            "category_l1",
            "num_transactions",
            "num_unique_brands",
            "brand_diversity_ratio"
        ])
    )



=== Random khách segment 0 ===
shape: (10, 5)
┌─────────────┬─────────────┬──────────────────┬───────────────────┬───────────────────────┐
│ customer_id ┆ category_l1 ┆ num_transactions ┆ num_unique_brands ┆ brand_diversity_ratio │
│ ---         ┆ ---         ┆ ---              ┆ ---               ┆ ---                   │
│ i32         ┆ str         ┆ u32              ┆ u32               ┆ f64                   │
╞═════════════╪═════════════╪══════════════════╪═══════════════════╪═══════════════════════╡
│ 1570091     ┆ Tã          ┆ 7                ┆ 2                 ┆ 0.285714              │
│ 5730926     ┆ Sữa         ┆ 10               ┆ 2                 ┆ 0.2                   │
│ 3801854     ┆ Tã          ┆ 11               ┆ 4                 ┆ 0.363636              │
│ 6102929     ┆ Tã          ┆ 8                ┆ 1                 ┆ 0.125                 │
│ 6497599     ┆ Tã          ┆ 6                ┆ 2                 ┆ 0.333333              │
│ 3072179     ┆ Tã     

In [ ]:

# Đường dẫn output
OUTPUT_DIR = r"D:\003. HK1 - Năm 3\02. CS116 - Python cho Máy học\CS116-DoAn\Phase-2\Feature engineering"
OUTPUT_PATH = os.path.join(OUTPUT_DIR, "brand_segment.parquet")

# Đảm bảo thư mục tồn tại
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Chỉ giữ các cột cần thiết và lưu
(
    brand_segment_df
    .select(["category_l1", "brand_segment"])
    .write_parquet(OUTPUT_PATH)
)

print("✅ Đã lưu brand_segment thành công tại:")
print(OUTPUT_PATH)

✅ Đã lưu brand_segment thành công tại:
D:\003. HK1 - Năm 3\02. CS116 - Python cho Máy học\CS116-DoAn\Phase-2\Feature engineering\brand_segment.parquet


### **Đặc trưng về hành vi mua hàng:** Xét hành vi mua hàng dựa trên độ tuổi của bé
Lấy độ đa dạng trong category_1, lấy top K sản phẩm tại mỗi loại để xem người dùng có bé trong độ tuổi đó thì thường mua những sản phẩm nào

### **Đặc trưng về hành vi mua hàng:** Thống kê những sản phẩm hay mua chung và số lần mua chung: item 1 | item 2 | #cooc (Cũng xét theo ngày)

### Discount user: Người dùng đó có mua hàng discount hay k. Tính dựa trên số lần mua trong tháng
Cũng có thể sẽ là gom cụm: mua ít, mua vừa, mua nhiều

### Đếm số lượng mặt hàng được bán ra (Mặt hàng được bán ra càng nhiều thì khả năng người ta mua hàng sẽ càng cao ) 
(Hệ số phổ biến của sản phẩm)

chuẩn hóa theo từng category_l1, và chỉ giữ TOP 10 item phổ biến nhất trong mỗi category_l1.

In [380]:

# =========================
# 1) TÍNH total_sold THEO item_id (gộp 20 chunk)
# =========================
TRANS_PATH = r"D:\003. HK1 - Năm 3\02. CS116 - Python cho Máy học\CS116-DoAn\Phase-2\preprocessing data"
trans_files = sorted(glob.glob(f"{TRANS_PATH}\\sale_pers.purchase_history_daily_chunk_*.parquet"))

item_sales = []

for file in trans_files:
    print("Processing:", file)

    df = (
        pl.read_parquet(file)
        .select(["item_id", "quantity"])
        .group_by("item_id")
        .agg(pl.col("quantity").sum().alias("total_sold"))
    )
    item_sales.append(df)

item_sales_all = (
    pl.concat(item_sales, how="vertical_relaxed")
    .group_by("item_id")
    .agg(pl.col("total_sold").sum().alias("total_sold"))
)

print("item_sales_all shape:", item_sales_all.shape)

# =========================
# 2) JOIN LẤY category_l1
# =========================
# df_item: item dataframe của bạn (đã load sẵn) phải có ["item_id","category_l1"]
item_basic = df_item.select(["item_id", "category_l1"])

item_sales_cat = (
    item_sales_all
    .join(item_basic, on="item_id", how="left")
    .filter(pl.col("category_l1").is_not_null())
)

print("item_sales_cat shape:", item_sales_cat.shape)

# =========================
# 3) TOP 10 item_id THEO MỖI category_l1
# =========================
top10_by_cat = (
    item_sales_cat
    .sort(["category_l1", "total_sold"], descending=[False, True])
    .group_by("category_l1")
    .head(10)
    .sort(["category_l1", "total_sold"], descending=[False, True])
)

print("\n✅ Top 10 item phổ biến nhất theo từng category_l1 (mẫu):")
print(top10_by_cat.head(30))

# =========================
# 4) CHECK: in ra 1 category bất kỳ để bạn nhìn rõ 10 item_id
# =========================
sample_cat = top10_by_cat.select("category_l1").unique().head(1).item()
print(f"\n=== CHECK TOP 10 của category_l1 = {sample_cat} ===")
print(
    top10_by_cat
    .filter(pl.col("category_l1") == sample_cat)
    .select(["category_l1", "item_id", "total_sold"])
)


Processing: D:\003. HK1 - Năm 3\02. CS116 - Python cho Máy học\CS116-DoAn\Phase-2\preprocessing data\sale_pers.purchase_history_daily_chunk_0.parquet
Processing: D:\003. HK1 - Năm 3\02. CS116 - Python cho Máy học\CS116-DoAn\Phase-2\preprocessing data\sale_pers.purchase_history_daily_chunk_1.parquet
Processing: D:\003. HK1 - Năm 3\02. CS116 - Python cho Máy học\CS116-DoAn\Phase-2\preprocessing data\sale_pers.purchase_history_daily_chunk_10.parquet
Processing: D:\003. HK1 - Năm 3\02. CS116 - Python cho Máy học\CS116-DoAn\Phase-2\preprocessing data\sale_pers.purchase_history_daily_chunk_11.parquet
Processing: D:\003. HK1 - Năm 3\02. CS116 - Python cho Máy học\CS116-DoAn\Phase-2\preprocessing data\sale_pers.purchase_history_daily_chunk_12.parquet
Processing: D:\003. HK1 - Năm 3\02. CS116 - Python cho Máy học\CS116-DoAn\Phase-2\preprocessing data\sale_pers.purchase_history_daily_chunk_13.parquet
Processing: D:\003. HK1 - Năm 3\02. CS116 - Python cho Máy học\CS116-DoAn\Phase-2\preprocessing 

In [384]:
top10_by_cat.head(50)

category_l1,item_id,total_sold
str,str,i32
"""Babycare""","""5950000000001""",203388
"""Babycare""","""0203000000004""",128333
"""Babycare""","""0007150000144""",123655
"""Babycare""","""0007150000031""",80181
"""Babycare""","""6498000000005""",60252
…,…,…
"""Sữa""","""2483000000004""",115008
"""Sữa""","""4355000000001""",99082
"""Sữa""","""6488000000001""",97755


In [385]:
# =========================
# SAVE top10_by_cat
# =========================

OUTPUT_DIR = r"D:\003. HK1 - Năm 3\02. CS116 - Python cho Máy học\CS116-DoAn\Phase-2\Feature engineering"
OUTPUT_PATH = f"{OUTPUT_DIR}\\top10_by_cat.parquet"

top10_by_cat.write_parquet(OUTPUT_PATH)

print("✅ Đã lưu top10_by_cat xuống:")
print(OUTPUT_PATH)
print("Shape:", top10_by_cat.shape)


✅ Đã lưu top10_by_cat xuống:
D:\003. HK1 - Năm 3\02. CS116 - Python cho Máy học\CS116-DoAn\Phase-2\Feature engineering\top10_by_cat.parquet
Shape: (140, 3)


**Xét Đếm số lượng mặt hàng được bán ra (Mặt hàng được bán ra càng nhiều thì khả năng người ta mua hàng sẽ càng cao ) (Hệ số phổ biến của sản phẩm), chuẩn hóa theo từng category_l1, và chỉ giữ TOP 10 item phổ biến nhất trong mỗi category_l1. Nhưng đếm theo tháng**

Tức là kiểu như tháng nào người dùng sẽ thường mua sản phẩm nào. Để sau này ví dụ thầy cho dữ liệu tháng 1/2025 thì có thể dùng những sản phẩm phổ biến của tháng 11/2024 để gợi ý

In [386]:

TRANS_PATH = r"D:\003. HK1 - Năm 3\02. CS116 - Python cho Máy học\CS116-DoAn\Phase-2\preprocessing data"
ITEM_PATH  = r"D:\003. HK1 - Năm 3\02. CS116 - Python cho Máy học\CS116-DoAn\Phase-2\Feature engineering\sale_pers.item_chunk_0.parquet"

trans_files = sorted(
    glob.glob(f"{TRANS_PATH}/sale_pers.purchase_history_daily_chunk_*.parquet")
)

item = (
    pl.read_parquet(ITEM_PATH)
    .select(["item_id", "category_l1"])
)

In [387]:
monthly_sales_chunks = []

for file in trans_files:
    print("Processing:", os.path.basename(file))

    df = (
        pl.read_parquet(file)
        .select(["item_id", "quantity", "created_date"])
        .with_columns(
            pl.col("created_date")
            .cast(pl.Date)
            .dt.strftime("%Y-%m")
            .alias("month")
        )
        .join(item, on="item_id", how="left")
        .filter(pl.col("category_l1").is_not_null())
        .group_by(["month", "category_l1", "item_id"])
        .agg(
            pl.col("quantity").sum().alias("total_sold")
        )
    )

    monthly_sales_chunks.append(df)


Processing: sale_pers.purchase_history_daily_chunk_0.parquet
Processing: sale_pers.purchase_history_daily_chunk_1.parquet
Processing: sale_pers.purchase_history_daily_chunk_10.parquet
Processing: sale_pers.purchase_history_daily_chunk_11.parquet
Processing: sale_pers.purchase_history_daily_chunk_12.parquet
Processing: sale_pers.purchase_history_daily_chunk_13.parquet
Processing: sale_pers.purchase_history_daily_chunk_14.parquet
Processing: sale_pers.purchase_history_daily_chunk_15.parquet
Processing: sale_pers.purchase_history_daily_chunk_16.parquet
Processing: sale_pers.purchase_history_daily_chunk_17.parquet
Processing: sale_pers.purchase_history_daily_chunk_18.parquet
Processing: sale_pers.purchase_history_daily_chunk_19.parquet
Processing: sale_pers.purchase_history_daily_chunk_2.parquet
Processing: sale_pers.purchase_history_daily_chunk_3.parquet
Processing: sale_pers.purchase_history_daily_chunk_4.parquet
Processing: sale_pers.purchase_history_daily_chunk_5.parquet
Processing: sa

In [388]:
monthly_sales_all = (
    pl.concat(monthly_sales_chunks, how="vertical_relaxed")
    .group_by(["month", "category_l1", "item_id"])
    .agg(
        pl.col("total_sold").sum().alias("total_sold")
    )
)

print("monthly_sales_all shape:", monthly_sales_all.shape)


monthly_sales_all shape: (153563, 4)


In [389]:
top10_by_cat_month = (
    monthly_sales_all
    .with_columns(
        pl.col("total_sold")
        .rank(method="dense", descending=True)
        .over(["month", "category_l1"])
        .alias("rank")
    )
    .filter(pl.col("rank") <= 10)
    .sort(["month", "category_l1", "rank"])
)

print("top10_by_cat_month shape:", top10_by_cat_month.shape)


top10_by_cat_month shape: (1691, 5)


In [390]:
print(
    top10_by_cat_month
    .filter(pl.col("month") == "2024-11")
    .head(50)
)


shape: (50, 5)
┌─────────┬─────────────┬───────────────┬────────────┬──────┐
│ month   ┆ category_l1 ┆ item_id       ┆ total_sold ┆ rank │
│ ---     ┆ ---         ┆ ---           ┆ ---        ┆ ---  │
│ str     ┆ str         ┆ str           ┆ i32        ┆ u32  │
╞═════════╪═════════════╪═══════════════╪════════════╪══════╡
│ 2024-11 ┆ Babycare    ┆ 5950000000001 ┆ 16759      ┆ 1    │
│ 2024-11 ┆ Babycare    ┆ 0203000000004 ┆ 10882      ┆ 2    │
│ 2024-11 ┆ Babycare    ┆ 0007150000144 ┆ 10481      ┆ 3    │
│ 2024-11 ┆ Babycare    ┆ 0007150000031 ┆ 6373       ┆ 4    │
│ 2024-11 ┆ Babycare    ┆ 6498000000005 ┆ 5682       ┆ 5    │
│ …       ┆ …           ┆ …             ┆ …          ┆ …    │
│ 2024-11 ┆ Sữa         ┆ 3774000000003 ┆ 13280      ┆ 5    │
│ 2024-11 ┆ Sữa         ┆ 4950000000001 ┆ 12142      ┆ 6    │
│ 2024-11 ┆ Sữa         ┆ 2483000000004 ┆ 9208       ┆ 7    │
│ 2024-11 ┆ Sữa         ┆ 2482000000004 ┆ 8949       ┆ 8    │
│ 2024-11 ┆ Sữa         ┆ 2578000000002 ┆ 8463       ┆ 

In [393]:
num_months = top10_by_cat_month.select("month").n_unique()

print(f"Số lượng tháng được lọc ra: {num_months}")


Số lượng tháng được lọc ra: 12


In [391]:
check = (
    top10_by_cat_month
    .group_by(["month", "category_l1"])
    .agg(pl.count().alias("num_items"))
    .filter(pl.col("num_items") != 10)
)

print("Các group KHÔNG đủ 10 item (nếu có):")
print(check)


Các group KHÔNG đủ 10 item (nếu có):
shape: (11, 3)
┌─────────┬────────────────────────┬───────────┐
│ month   ┆ category_l1            ┆ num_items │
│ ---     ┆ ---                    ┆ ---       │
│ str     ┆ str                    ┆ u32       │
╞═════════╪════════════════════════╪═══════════╡
│ 2024-12 ┆ Thực phẩm cho bé       ┆ 11        │
│ 2024-08 ┆ Thời trang             ┆ 11        │
│ 2024-09 ┆ Tã                     ┆ 11        │
│ 2024-07 ┆ Thực phẩm cho gia đình ┆ 11        │
│ 2024-03 ┆ Thực phẩm cho gia đình ┆ 11        │
│ …       ┆ …                      ┆ …         │
│ 2024-06 ┆ Hóa mỹ phẩm gia đình   ┆ 11        │
│ 2024-11 ┆ Phụ kiện               ┆ 11        │
│ 2024-10 ┆ Tã                     ┆ 11        │
│ 2024-10 ┆ Phụ kiện               ┆ 11        │
│ 2024-01 ┆ Phụ kiện               ┆ 11        │
└─────────┴────────────────────────┴───────────┘


C:\Users\PC\AppData\Local\Temp\ipykernel_12820\1899442887.py:4: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  .agg(pl.count().alias("num_items"))


In [392]:
OUTPUT_DIR = r"D:\003. HK1 - Năm 3\02. CS116 - Python cho Máy học\CS116-DoAn\Phase-2\Feature engineering"
OUTPUT_PATH = f"{OUTPUT_DIR}\\top10_by_cat_month.parquet"

top10_by_cat_month.write_parquet(OUTPUT_PATH)

print("✅ Đã lưu top10_by_cat_month")
print("Path:", OUTPUT_PATH)


✅ Đã lưu top10_by_cat_month
Path: D:\003. HK1 - Năm 3\02. CS116 - Python cho Máy học\CS116-DoAn\Phase-2\Feature engineering\top10_by_cat_month.parquet


### Lần cuối cùng mua hàng của khách hàng

In [ ]:
# Như tạo r biến: Recency

### =========================================================================================

In [400]:
pl.Config.set_tbl_rows(100)

polars.config.Config